# 10 — FAISS Metric Comparison

Build three FAISS indices over the same embeddings and compare top-5 results.

| Metric | Index | Query normalisation | Score direction |
|---|---|---|---|
| Cosine | `IndexFlatIP` | L2-normalize query | ↑ higher better |
| Inner Product | `IndexFlatIP` | none | ↑ higher better |
| Euclidean | `IndexFlatL2` | none | ↓ lower better |

**Config source:** `configs/default.yaml` → `data`, `splitting`, `embeddings`

In [ ]:
import faiss
import numpy as np

from rag_pipeline.utils import load_notebook_config
from rag_pipeline.embeddings import build_embeddings
from rag_pipeline.ingestion import load_documents
from rag_pipeline.splitting import split_documents

cfg, REPO = load_notebook_config()
QUERIES = cfg.notebooks["retrieval_queries"]

**Load + chunk + embed**

In [ ]:
docs = load_documents(
    cfg.data["sources"],
    sample_fraction=cfg.data.get("sample_fraction"),
    sample_size=cfg.data.get("sample_size"),
    sample_seed=cfg.data.get("sample_seed", 42),
)[:500]
chunks = split_documents(docs, dict(cfg.splitting))
print(f"Chunks: {len(chunks)}")

emb = build_embeddings(dict(cfg.embeddings))
matrix = np.asarray(emb.embed_documents([c.page_content for c in chunks]), dtype=np.float32)
print(f"Embedding matrix: {matrix.shape}")

**Build three indices**

In [ ]:
dim = matrix.shape[1]

# Cosine: normalize then IP
matrix_cos = np.ascontiguousarray(matrix.copy())
faiss.normalize_L2(matrix_cos)
index_cos = faiss.IndexFlatIP(dim)
index_cos.add(matrix_cos)

# Inner product: raw
matrix_ip = np.ascontiguousarray(matrix.copy())
index_ip = faiss.IndexFlatIP(dim)
index_ip.add(matrix_ip)

# Euclidean: raw
matrix_l2 = np.ascontiguousarray(matrix.copy())
index_l2 = faiss.IndexFlatL2(dim)
index_l2.add(matrix_l2)

print(f"cosine={index_cos.ntotal} | ip={index_ip.ntotal} | l2={index_l2.ntotal}")

**Query all three**

In [ ]:
INDEXES = {
    "COSINE (norm→IP)": {"index": index_cos, "normalize": True,  "direction": "↑"},
    "INNER PRODUCT":    {"index": index_ip,  "normalize": False, "direction": "↑"},
    "EUCLIDEAN (L2)":   {"index": index_l2,  "normalize": False, "direction": "↓"},
}

def prep(q, normalize):
    v = np.ascontiguousarray(np.asarray(emb.embed_query(q), dtype=np.float32).reshape(1, -1))
    if normalize:
        faiss.normalize_L2(v)
    return v

for q in QUERIES[:3]:
    print(f"\n{'='*80}\nQuery: {q}\n{'='*80}")
    for name, spec in INDEXES.items():
        distances, indices = spec["index"].search(prep(q, spec["normalize"]), k=5)
        print(f"\n{name}")
        print("-" * 78)
        for rank, (dist, idx) in enumerate(zip(distances[0], indices[0]), 1):
            preview = chunks[idx].page_content.replace("\n", " ")[:70]
            row = chunks[idx].metadata.get("row", idx)
            print(f"  {rank}. row={row:>4}  {spec['direction']} {dist:>8.4f}  {preview}...")